# Erro Percentual e Visualização — Método Aritmético (Linear)

## Importações

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import plotly.graph_objects as go

from utilitarios_populacional import (
    COLUNA_MUNICIPIO,
    COLUNA_CODIGO_IBGE,
    COLUNA_REGIONAL,
    normalizar_codigo_ibge,
    calcular_taxa_aritmetica,
    estimar_populacao_aritmetica,
    resumir_ep,
    agregar_ep_estadual,
)

## Configuração

In [2]:
CAMINHO_ARQUIVO_ENTRADA = Path('/home/anna/tcc/Seleção e Transformação/População - Faixa Etária/tabela_populacao_faixa_etaria.csv')
CAMINHO_ARQUIVO_SAIDA = Path('/home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/populacao_interpolacao_extrapolacao_linear.csv')
CAMINHO_REGIONAIS = Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv")

LISTA_ANOS_CENSITARIOS = [1970, 1980, 1991, 2000, 2010, 2022]
ANO_INICIO = 1970
ANO_FIM = 2025
LISTA_ANOS_COMPLETOS = list(range(ANO_INICIO, ANO_FIM + 1))
ANOS_A_EXTRAPOLAR = [ano for ano in LISTA_ANOS_COMPLETOS if ano > LISTA_ANOS_CENSITARIOS[-1]]

FIG_WIDTH, FIG_HEIGHT = 1100, 550
FONTE_FAMILIA = "Arial"
FONTE_TITULO = 18
FONTE_EIXO = 13

COR_CENSO = "#1B1B1B"
COR_INTERPOLACAO = "#2E5C8A"
COR_EXTRAPOLACAO = "#C0392B"
COR_EP_INTERPOLACAO = "#2E5C8A"
COR_EP_EXTRAPOLACAO = "#C0392B"
COR_LIMITE_10PCT = "#999999"
COR_BARRAS_REGIONAL = "#2E5C8A"

TITULO_GRAFICO_POPULACAO = "População Total de Minas Gerais — Interpolação e Extrapolação (Método Aritmético)"
TITULO_GRAFICO_ERRO = "Erro Percentual (EP) do Estado — Anos Censitários Testados (Método Aritmético)"
TITULO_GRAFICO_REGIONAL = "População Feminina por Regional de Saúde — Minas Gerais (Método Aritmético)"

EIXO_X_POPULACAO = "Ano"
EIXO_Y_POPULACAO = "População (Minas Gerais)"
EIXO_X_ERRO = "Ano testado"
EIXO_Y_ERRO = "Erro Percentual — EP (%)"
EIXO_X_REGIONAL = "Regional de Saúde"
EIXO_Y_REGIONAL = "População Feminina"

ANO_REFERENCIA_GRAFICO_REGIONAL = 2025

## Carregamento dos Dados

In [3]:
tabela_populacao = pd.read_csv(CAMINHO_ARQUIVO_ENTRADA)
tabela_populacao[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_populacao[COLUNA_CODIGO_IBGE])
lista_colunas_totais = [f"TOTAL_{ano}" for ano in LISTA_ANOS_CENSITARIOS]
tabela_totais = tabela_populacao[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE] + lista_colunas_totais].copy()

tabela_saida = pd.read_csv(CAMINHO_ARQUIVO_SAIDA)
tabela_saida[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_saida[COLUNA_CODIGO_IBGE])

tabela_regionais = pd.read_csv(CAMINHO_REGIONAIS)
tabela_regionais[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_regionais[COLUNA_CODIGO_IBGE])

codigos_saida_bruto = set(tabela_saida[COLUNA_CODIGO_IBGE])
codigos_regionais_bruto = set(tabela_regionais[COLUNA_CODIGO_IBGE])
diferenca = codigos_saida_bruto - codigos_regionais_bruto
if diferenca:
    print(f"ATENÇÃO: {len(diferenca)} código(s) IBGE de tabela_saida sem correspondência em tabela_regionais: {sorted(diferenca, key=str)}")
else:
    print("Todos os municípios de tabela_saida têm correspondência em tabela_regionais.")

mapa_regional = tabela_regionais.set_index(COLUNA_CODIGO_IBGE)[COLUNA_REGIONAL]

print(f"tabela_totais    : {tabela_totais.shape[0]} municípios")
print(f"tabela_saida     : {tabela_saida.shape[0]} municípios, {ANO_INICIO}-{ANO_FIM}")
print(f"tabela_regionais : {tabela_regionais.shape[0]} municípios mapeados")

Todos os municípios de tabela_saida têm correspondência em tabela_regionais.
tabela_totais    : 853 municípios
tabela_saida     : 853 municípios, 1970-2025
tabela_regionais : 853 municípios mapeados


## Erro Percentual da Interpolação

In [4]:
CENSOS_DO_MEIO = LISTA_ANOS_CENSITARIOS[1:-1]  # [1980, 1991, 2000, 2010]

lista_ep_interpolacao = []

for ano_escondido in CENSOS_DO_MEIO:
    censos_restantes = [ano for ano in LISTA_ANOS_CENSITARIOS if ano != ano_escondido]
    ano_anterior = max(ano for ano in censos_restantes if ano < ano_escondido)
    ano_posterior = min(ano for ano in censos_restantes if ano > ano_escondido)

    populacao_anterior = tabela_totais[f"TOTAL_{ano_anterior}"]
    populacao_posterior = tabela_totais[f"TOTAL_{ano_posterior}"]
    tempo_total = ano_posterior - ano_anterior
    tempo_decorrido = ano_escondido - ano_anterior

    k_a = calcular_taxa_aritmetica(populacao_anterior, populacao_posterior, tempo_total)
    populacao_estimada = estimar_populacao_aritmetica(populacao_anterior, k_a, tempo_decorrido)
    populacao_observada = tabela_totais[f"TOTAL_{ano_escondido}"]

    # Município com população observada = 0 (ex.: ainda não emancipado naquele
    # censo) torna o EP matematicamente indefinido (divisão por zero). Ao
    # contrário do método geométrico — que já produz NaN nesses casos por
    # causa da guarda em calcular_taxa_geometrica —, o método aritmético
    # estima um valor numérico normalmente, então o `inf` resultante precisa
    # ser convertido para NaN aqui, senão contamina médias/desvio-padrão
    # agregados mais adiante (inf não é removido por dropna()).
    populacao_observada_valida = populacao_observada.where(populacao_observada > 0)
    ep = ((populacao_estimada - populacao_observada_valida) / populacao_observada_valida) * 100

    lista_ep_interpolacao.append(pd.DataFrame({
        COLUNA_MUNICIPIO: tabela_totais[COLUNA_MUNICIPIO],
        COLUNA_CODIGO_IBGE: tabela_totais[COLUNA_CODIGO_IBGE],
        "ANO_TESTADO": ano_escondido,
        "TIPO": "Interpolação",
        "CENSOS_USADOS": f"{ano_anterior}-{ano_posterior}",
        "POPULACAO_OBSERVADA": populacao_observada,
        "POPULACAO_ESTIMADA": populacao_estimada,
        "EP": ep,
    }))

tabela_ep_interpolacao = pd.concat(lista_ep_interpolacao, ignore_index=True)

quantidade_ep_indefinido = tabela_ep_interpolacao["EP"].isna().sum()
if quantidade_ep_indefinido > 0:
    print(f"ATENÇÃO: {quantidade_ep_indefinido} linha(s) com população observada = 0 "
          f"(EP indefinido, excluído como NaN).")

tabela_ep_interpolacao.head()

ATENÇÃO: 130 linha(s) com população observada = 0 (EP indefinido, excluído como NaN).


,MUNICÍPIOS,CÓDIGO IBGE,ANO_TESTADO,TIPO,CENSOS_USADOS,POPULACAO_OBSERVADA,POPULACAO_ESTIMADA,EP
0,abadia dos dourados,3100104,1980,Interpolação,1970-1991,3936,4088.000000,3.861789
1,abaete,3100203,1980,Interpolação,1970-1991,9926,10374.571429,4.519156
2,abre campo,3100302,1980,Interpolação,1970-1991,9375,9470.428571,1.017905
3,acaiaca,3100401,1980,Interpolação,1970-1991,1822,1812.333333,-0.530553
4,aguanil,3100807,1980,Interpolação,1970-1991,1560,1437.714286,-7.838828


## Erro Percentual da Extrapolação

In [5]:
PARES_EXTRAPOLACAO_TESTE = [
    (1991, 2000, 2010),
    (2000, 2010, 2022),
]

lista_ep_extrapolacao = []

for ano_inicial, ano_final, ano_alvo in PARES_EXTRAPOLACAO_TESTE:
    populacao_inicial = tabela_totais[f"TOTAL_{ano_inicial}"]
    populacao_final = tabela_totais[f"TOTAL_{ano_final}"]
    tempo_total = ano_final - ano_inicial
    tempo_decorrido = ano_alvo - ano_final

    k_a = calcular_taxa_aritmetica(populacao_inicial, populacao_final, tempo_total)
    populacao_estimada = estimar_populacao_aritmetica(populacao_final, k_a, tempo_decorrido)
    populacao_observada = tabela_totais[f"TOTAL_{ano_alvo}"]

    # Mesma guarda contra divisão por zero da célula de interpolação acima.
    populacao_observada_valida = populacao_observada.where(populacao_observada > 0)
    ep = ((populacao_estimada - populacao_observada_valida) / populacao_observada_valida) * 100

    lista_ep_extrapolacao.append(pd.DataFrame({
        COLUNA_MUNICIPIO: tabela_totais[COLUNA_MUNICIPIO],
        COLUNA_CODIGO_IBGE: tabela_totais[COLUNA_CODIGO_IBGE],
        "ANO_TESTADO": ano_alvo,
        "TIPO": "Extrapolação",
        "CENSOS_USADOS": f"{ano_inicial}-{ano_final}",
        "POPULACAO_OBSERVADA": populacao_observada,
        "POPULACAO_ESTIMADA": populacao_estimada,
        "EP": ep,
    }))

tabela_ep_extrapolacao = pd.concat(lista_ep_extrapolacao, ignore_index=True)

quantidade_ep_indefinido = tabela_ep_extrapolacao["EP"].isna().sum()
if quantidade_ep_indefinido > 0:
    print(f"ATENÇÃO: {quantidade_ep_indefinido} linha(s) com população observada = 0 "
          f"(EP indefinido, excluído como NaN).")

tabela_ep_extrapolacao.head()

,MUNICÍPIOS,CÓDIGO IBGE,ANO_TESTADO,TIPO,CENSOS_USADOS,POPULACAO_OBSERVADA,POPULACAO_ESTIMADA,EP
0,abadia dos dourados,3100104,2010,Extrapolação,1991-2000,3676,2941.666667,-19.976424
1,abaete,3100203,2010,Extrapolação,1991-2000,11171,11377.000000,1.844061
2,abre campo,3100302,2010,Extrapolação,1991-2000,7238,3808.888889,-47.376501
3,acaiaca,3100401,2010,Extrapolação,1991-2000,1881,1806.111111,-3.981334
4,aguanil,3100807,2010,Extrapolação,1991-2000,1938,1949.666667,0.601995


## Resumo do Erro

In [6]:
tabela_resumo_ep = pd.DataFrame([
    resumir_ep(tabela_ep_interpolacao, "Interpolação"),
    resumir_ep(tabela_ep_extrapolacao, "Extrapolação"),
])

tabela_resumo_ep

,Validação,EP médio (%),EP mediano (%),MAPE (%),Desvio-padrão (%)
0,Interpolação,9.997,0.136,19.274,56.240
1,Extrapolação,5.597,-1.009,24.521,40.749


## Agregação Estadual — Erro

In [7]:
tabela_ep_estadual = pd.concat([
    agregar_ep_estadual(tabela_ep_interpolacao),
    agregar_ep_estadual(tabela_ep_extrapolacao),
], ignore_index=True).sort_values("ANO_TESTADO")

tabela_ep_estadual

,ANO_TESTADO,TIPO,POPULACAO_OBSERVADA,POPULACAO_ESTIMADA,EP_ESTADO
0,1980,Interpolação,7339191,6.998043e+06,-4.648310
1,1991,Interpolação,8346023,8.393063e+06,0.563622
2,2000,Interpolação,9255322,8.768020e+06,-5.265098
3,2010,Interpolação,9236906,9.832121e+06,6.443879
4,2010,Extrapolação,9236906,1.026565e+07,11.137368
5,2022,Extrapolação,10524280,9.214807e+06,-12.442402


## Agregação Estadual — População

In [8]:
colunas_ano_completas = [f"TOTAL_{ano}" for ano in LISTA_ANOS_COMPLETOS]
populacao_estadual_completa = tabela_saida[colunas_ano_completas].sum()
populacao_estadual_completa.index = LISTA_ANOS_COMPLETOS

populacao_estadual_censos = tabela_totais[lista_colunas_totais].sum()
populacao_estadual_censos.index = LISTA_ANOS_CENSITARIOS

populacao_estadual_censos

1970     5772606
1980     7339191
1991     8346023
2000     9255322
2010     9236906
2022    10524280
dtype: int64

## População por Regional de Saúde

In [9]:
tabela_saida_com_regional = tabela_saida.copy()
tabela_saida_com_regional[COLUNA_REGIONAL] = tabela_saida_com_regional[COLUNA_CODIGO_IBGE].map(mapa_regional)

colunas_totais_completas = [f"TOTAL_{ano}" for ano in LISTA_ANOS_COMPLETOS]

tabela_populacao_por_regional = (
    tabela_saida_com_regional
    .groupby(COLUNA_REGIONAL, as_index=False)[colunas_totais_completas]
    .sum()
)

tabela_populacao_por_regional.head()

,REGIONAL NORMALIZADA,TOTAL_1970,TOTAL_1971,TOTAL_1972,TOTAL_1973,TOTAL_1974,TOTAL_1975,TOTAL_1976,TOTAL_1977,TOTAL_1978,...,TOTAL_2016,TOTAL_2017,TOTAL_2018,TOTAL_2019,TOTAL_2020,TOTAL_2021,TOTAL_2022,TOTAL_2023,TOTAL_2024,TOTAL_2025
0,alfenas,139443,142904,146349,149796,153244,156687,160138,163589,167033,...,223989,225921,227846,229775,231704,233632,235547,237487,239413,241340
1,barbacena,156471,160584,164677,168774,172870,176963,181063,185163,189256,...,247861,250145,252419,254695,256976,259254,261518,263810,266087,268360
2,belo horizonte,951683,1017583,1083469,1149356,1215241,1281122,1347014,1412902,1478786,...,2560035,2598430,2636814,2675203,2713588,2751975,2790345,2828752,2867139,2905522
3,coronel fabriciano,218147,227359,236564,245770,254976,264177,273386,282593,291798,...,387487,390588,393682,396775,399873,402969,406050,409163,412257,415350
4,diamantina,173917,175723,177514,179308,181100,182886,184683,186478,188269,...,200612,200226,199832,199448,199059,198671,198264,197886,197495,197106


## Gráfico 1 — Curva da População do Estado

In [10]:
anos_interpolados = [ano for ano in LISTA_ANOS_COMPLETOS if ano <= LISTA_ANOS_CENSITARIOS[-1]]
populacao_interpolada = [populacao_estadual_completa[ano] for ano in anos_interpolados]

anos_extrapolados = [ano for ano in LISTA_ANOS_COMPLETOS if ano >= LISTA_ANOS_CENSITARIOS[-1]]
populacao_extrapolada = [populacao_estadual_completa[ano] for ano in anos_extrapolados]

populacao_censos = [populacao_estadual_censos[ano] for ano in LISTA_ANOS_CENSITARIOS]
populacao_extrapolados_destaque = [populacao_estadual_completa[ano] for ano in ANOS_A_EXTRAPOLAR]

fig_populacao = go.Figure()

fig_populacao.add_trace(go.Scatter(
    x=anos_interpolados, y=populacao_interpolada,
    mode="lines", name="Interpolação (entre censos)",
    line=dict(color=COR_INTERPOLACAO, width=2.5),
    hovertemplate="Ano: %{x}<br>População: %{y:,.0f}<extra></extra>",
))

fig_populacao.add_trace(go.Scatter(
    x=anos_extrapolados, y=populacao_extrapolada,
    mode="lines", name="Extrapolação (após o último censo)",
    line=dict(color=COR_EXTRAPOLACAO, width=2.5, dash="dash"),
    hovertemplate="Ano: %{x}<br>População: %{y:,.0f}<extra></extra>",
))

fig_populacao.add_trace(go.Scatter(
    x=LISTA_ANOS_CENSITARIOS, y=populacao_censos,
    mode="markers", name="Censos (observado)",
    marker=dict(color=COR_CENSO, size=11),
    hovertemplate="Censo %{x}<br>População: %{y:,.0f}<extra></extra>",
))

fig_populacao.add_trace(go.Scatter(
    x=ANOS_A_EXTRAPOLAR, y=populacao_extrapolados_destaque,
    mode="markers", name="Anos extrapolados",
    marker=dict(color=COR_EXTRAPOLACAO, size=9, symbol="diamond"),
    hovertemplate="Ano: %{x}<br>População: %{y:,.0f}<extra></extra>",
))

fig_populacao.update_layout(
    title=dict(text=TITULO_GRAFICO_POPULACAO, x=0.5, xanchor="center",
               font=dict(size=FONTE_TITULO, family=FONTE_FAMILIA)),
    xaxis_title=EIXO_X_POPULACAO, yaxis_title=EIXO_Y_POPULACAO,
    font=dict(family=FONTE_FAMILIA, size=FONTE_EIXO),
    width=FIG_WIDTH, height=FIG_HEIGHT,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.22, x=0.5, xanchor="center"),
    plot_bgcolor="white", paper_bgcolor="white",
)

fig_populacao.update_xaxes(
    showgrid=True, gridcolor="#E5E5E5",
    tickmode="array",
    tickvals=sorted(set(LISTA_ANOS_CENSITARIOS + ANOS_A_EXTRAPOLAR)),
    tickangle=-45,
)
fig_populacao.update_yaxes(showgrid=True, gridcolor="#E5E5E5", tickformat=",.0f")

fig_populacao.show()

## Gráfico 2 — Erro Percentual do Estado

In [11]:
fig_erro = go.Figure()

for tipo, cor in [("Interpolação", COR_EP_INTERPOLACAO), ("Extrapolação", COR_EP_EXTRAPOLACAO)]:
    subset = tabela_ep_estadual[tabela_ep_estadual["TIPO"] == tipo]
    fig_erro.add_trace(go.Bar(
        x=subset["ANO_TESTADO"], y=subset["EP_ESTADO"],
        name=tipo, marker=dict(color=cor),
        text=[f"{valor:.2f}%" for valor in subset["EP_ESTADO"]],
        textposition="outside",
        hovertemplate="Ano testado: %{x}<br>EP: %{y:.2f}%<extra></extra>",
    ))

fig_erro.add_hline(y=10, line=dict(color=COR_LIMITE_10PCT, width=1, dash="dot"),
                    annotation_text="+10%", annotation_position="top right")
fig_erro.add_hline(y=-10, line=dict(color=COR_LIMITE_10PCT, width=1, dash="dot"),
                    annotation_text="-10%", annotation_position="bottom right")
fig_erro.add_hline(y=0, line=dict(color="#444444", width=1))

fig_erro.update_layout(
    title=dict(text=TITULO_GRAFICO_ERRO, x=0.5, xanchor="center",
               font=dict(size=FONTE_TITULO, family=FONTE_FAMILIA)),
    xaxis_title=EIXO_X_ERRO, yaxis_title=EIXO_Y_ERRO,
    font=dict(family=FONTE_FAMILIA, size=FONTE_EIXO),
    width=FIG_WIDTH, height=FIG_HEIGHT,
    barmode="group",
    legend=dict(orientation="h", y=-0.22, x=0.5, xanchor="center"),
    plot_bgcolor="white", paper_bgcolor="white",
)

fig_erro.update_xaxes(
    showgrid=True, gridcolor="#E5E5E5",
    tickmode="array",
    tickvals=sorted(tabela_ep_estadual["ANO_TESTADO"].unique()),
)
fig_erro.update_yaxes(showgrid=True, gridcolor="#E5E5E5")

fig_erro.show()

## Gráfico 3 — População por Regional de Saúde

In [12]:
coluna_ano_referencia = f"TOTAL_{ANO_REFERENCIA_GRAFICO_REGIONAL}"

tabela_grafico_regional = tabela_populacao_por_regional[[COLUNA_REGIONAL, coluna_ano_referencia]].copy()
tabela_grafico_regional = tabela_grafico_regional.sort_values(coluna_ano_referencia, ascending=False)

nomes_das_regionais = [nome.title() for nome in tabela_grafico_regional[COLUNA_REGIONAL]]

fig_populacao_regional = go.Figure()

fig_populacao_regional.add_trace(go.Bar(
    x=nomes_das_regionais,
    y=tabela_grafico_regional[coluna_ano_referencia],
    marker=dict(color=COR_BARRAS_REGIONAL),
    text=[f"{valor:,.0f}" for valor in tabela_grafico_regional[coluna_ano_referencia]],
    textposition="outside",
    hovertemplate="%{x}<br>População: %{y:,.0f}<extra></extra>",
))

fig_populacao_regional.update_layout(
    title=dict(text=f"{TITULO_GRAFICO_REGIONAL} ({ANO_REFERENCIA_GRAFICO_REGIONAL})", x=0.5, xanchor="center",
               font=dict(size=FONTE_TITULO, family=FONTE_FAMILIA)),
    xaxis_title=EIXO_X_REGIONAL, yaxis_title=EIXO_Y_REGIONAL,
    font=dict(family=FONTE_FAMILIA, size=FONTE_EIXO),
    width=FIG_WIDTH + 200, height=FIG_HEIGHT,
    plot_bgcolor="white", paper_bgcolor="white",
)

fig_populacao_regional.update_xaxes(tickangle=-45)
fig_populacao_regional.update_yaxes(showgrid=True, gridcolor="#E5E5E5", tickformat=",.0f")

fig_populacao_regional.show()